# 🚗 Autonomous Driving Perception System (flagship)
Combines YOLO26 + ByteTrack (vehicle + pedestrian detection/tracking), Depth Anything (per-object distance), and a HUD side panel with escalating collision-risk warnings -- the same recipe real driver-assist systems use, built from pieces across your series (Speed Estimator, Depth Vision, Sixth Sense).

In [1]:
!pip install ultralytics transformers accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.4 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import cv2
from PIL import Image
from collections import defaultdict, deque
from ultralytics import YOLO
from transformers import pipeline
from google.colab import files

yolo_model = YOLO("yolo26x.pt")

device = 0 if torch.cuda.is_available() else -1
depth_pipe = pipeline(task="depth-estimation",
                       model="depth-anything/Depth-Anything-V2-Small-hf",  # Small for speed -- two models run per frame here
                       device=device)
print("Models loaded.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 99.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Models loaded.


## Step 1: Upload your dashcam footage

In [3]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"FPS: {fps}, Resolution: {w}x{h}, Duration: {total_frames/fps:.1f}s")

Saving Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4 to Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4
FPS: 30.0, Resolution: 720x1280, Duration: 54.2s


## Step 2: Configuration + calibration
Depth is relative (not real metric units), so -- same as your Depth Vision project -- pick two rough distance anchors by eye: how far is the closest visible thing, and how far is the farthest?

In [4]:
VEHICLE_CLASSES = {2: "Car", 3: "Motorcycle", 5: "Bus", 7: "Truck"}
PERSON_CLASS = 0
DETECT_CLASSES = [PERSON_CLASS] + list(VEHICLE_CLASSES.keys())

VEHICLE_COLOR = (0, 140, 255)     # orange, matches the reference image
PERSON_COLOR = (255, 220, 0)      # cyan

# Distance calibration (rough eyeball anchors, same approach as Depth Vision)
NEAR_DISTANCE_M = 2
FAR_DISTANCE_M = 60

def depth_value_to_distance_m(depth_value_0_255):
    frac = depth_value_0_255 / 255.0
    return FAR_DISTANCE_M + (NEAR_DISTANCE_M - FAR_DISTANCE_M) * frac

# --- Collision risk thresholds ---
SAFE_DISTANCE_M = 20
WARNING_DISTANCE_M = 8
FAST_CLOSING_SPEED_MPS = 8   # closing speed that escalates CAUTION to WARNING early

DISTANCE_HISTORY_LEN = 6
SKIP_DEPTH_FRAMES = 1   # raise if too slow; YOLO still tracks every frame regardless

## Step 3: Distance, closing speed, and risk logic

In [5]:
distance_history = defaultdict(lambda: deque(maxlen=DISTANCE_HISTORY_LEN))

def get_object_depth_value(depth_map, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(depth_map.shape[1], x2), min(depth_map.shape[0], y2)
    if x2 <= x1 or y2 <= y1:
        return 128
    region = depth_map[y1:y2, x1:x2]
    return float(np.median(region)) if region.size > 0 else 128

def update_distance_history(track_id, distance_m):
    distance_history[track_id].append(distance_m)

def get_closing_speed(track_id, fps):
    hist = distance_history[track_id]
    if len(hist) < 2:
        return 0.0
    n_frames = len(hist) - 1
    delta_m = hist[0] - hist[-1]
    time_s = n_frames / fps
    return delta_m / time_s if time_s > 0 else 0.0

def risk_level(distance_m, closing_speed_mps):
    if distance_m <= WARNING_DISTANCE_M:
        return "WARNING"
    if distance_m <= SAFE_DISTANCE_M:
        if closing_speed_mps > FAST_CLOSING_SPEED_MPS:
            return "WARNING"
        return "CAUTION"
    return "SAFE"

RISK_COLORS = {"SAFE": (0, 200, 0), "CAUTION": (0, 165, 255), "WARNING": (0, 0, 255)}
RISK_RANK = {"SAFE": 0, "CAUTION": 1, "WARNING": 2}

## Step 4: Drawing — boxes, HUD panel

In [6]:
def draw_object_box(frame, box, label, distance_m, color):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    text = f"{label}"
    if distance_m is not None:
        text += f" {distance_m:.1f}m"
    cv2.putText(frame, text, (x1, y1 - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

def draw_hud_panel(frame, vehicle_count, person_count, closest_distance, overall_risk, flash_on):
    panel_w = 170
    x0 = w - panel_w
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, 0), (w, h), (10, 10, 10), -1)
    frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)

    cv2.putText(frame, "AUTONOMY HUD", (x0 + 10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    cv2.line(frame, (x0 + 10, 40), (w - 10, 40), (80, 80, 80), 1)

    cv2.putText(frame, "VEHICLES", (x0 + 10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1)
    cv2.putText(frame, str(vehicle_count), (x0 + 10, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.8, VEHICLE_COLOR, 2)

    cv2.putText(frame, "PEDESTRIANS", (x0 + 10, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1)
    cv2.putText(frame, str(person_count), (x0 + 10, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, PERSON_COLOR, 2)

    cv2.putText(frame, "CLOSEST", (x0 + 10, 180), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (180, 180, 180), 1)
    closest_text = f"{closest_distance:.1f}m" if closest_distance is not None else "--"
    cv2.putText(frame, closest_text, (x0 + 10, 205), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    risk_color = RISK_COLORS[overall_risk]
    if overall_risk == "WARNING" and not flash_on:
        risk_color = (255, 255, 255)
    cv2.rectangle(frame, (x0 + 10, h - 60), (w - 10, h - 20), risk_color, -1)
    cv2.putText(frame, overall_risk, (x0 + 25, h - 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
    return frame

## Step 5: Main processing loop
Runs YOLO tracking every frame and the depth model every `SKIP_DEPTH_FRAMES` frames (holding the last depth map in between) since it's the heavier of the two.

In [7]:
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
out = cv2.VideoWriter("driving_perception_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

frame_idx = 0
last_depth_map = None

results_gen = yolo_model.track(video_path, tracker="bytetrack.yaml", stream=True, classes=DETECT_CLASSES)

for result in results_gen:
    frame = result.orig_img

    if frame_idx % SKIP_DEPTH_FRAMES == 0:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_frame = Image.fromarray(frame_rgb)
        depth_result = depth_pipe(pil_frame)
        depth_map = np.array(depth_result["depth"])
        if depth_map.shape[:2] != (h, w):
            depth_map = cv2.resize(depth_map, (w, h))
        last_depth_map = depth_map
    else:
        depth_map = last_depth_map if last_depth_map is not None else np.full((h, w), 128, dtype=np.uint8)

    vehicle_count = 0
    person_count = 0
    closest_distance = None
    overall_risk = "SAFE"

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        ids = result.boxes.id.cpu().numpy().astype(int)
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)

        for box, track_id, cls_id in zip(boxes, ids, cls_ids):
            depth_value = get_object_depth_value(depth_map, box)
            distance_m = depth_value_to_distance_m(depth_value)
            update_distance_history(track_id, distance_m)
            closing_speed = get_closing_speed(track_id, fps)
            risk = risk_level(distance_m, closing_speed)

            if closest_distance is None or distance_m < closest_distance:
                closest_distance = distance_m
            if RISK_RANK[risk] > RISK_RANK[overall_risk]:
                overall_risk = risk

            if cls_id == PERSON_CLASS:
                person_count += 1
                label = "Person"
                base_color = PERSON_COLOR
            else:
                vehicle_count += 1
                label = VEHICLE_CLASSES.get(cls_id, "Vehicle")
                base_color = VEHICLE_COLOR

            box_color = RISK_COLORS[risk] if risk != "SAFE" else base_color
            draw_object_box(frame, box, label, distance_m, box_color)

    flash_on = (frame_idx // int(fps / 4 + 1)) % 2 == 0
    frame = draw_hud_panel(frame, vehicle_count, person_count, closest_distance, overall_risk, flash_on)

    out.write(frame)
    if frame_idx % 30 == 0:
        print(f"Processed frame {frame_idx}/{total_frames}")
    frame_idx += 1

out.release()
print("Done -> driving_perception_output.mp4")

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 256ms
Prepared 1 package in 36ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


video 1/1 (frame 1/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 9 cars, 3 trucks, 61.1ms
Processed frame 0/1626
video 1/1 (frame 2/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 9 cars, 3 trucks, 41.2ms
video 1/1 (frame 3/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 7 cars, 3 trucks, 41.2ms
video 1/1 (frame 4/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 7 cars, 3 trucks, 30.0ms
vid

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


video 1/1 (frame 12/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 6 cars, 4 trucks, 34.9ms
video 1/1 (frame 13/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 6 cars, 3 trucks, 29.4ms
video 1/1 (frame 14/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 5 cars, 5 trucks, 29.1ms
video 1/1 (frame 15/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 7 cars, 4 trucks, 28.7ms
video 1/1 (frame 16/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 7 cars, 3 trucks, 28.8ms
video 1/1 (frame 17/1626) /content/Driving in Dubai – How to Exit Dubai Roads Safely _ Smart_  _rtadubaigov(720P_HD).mp4: 640x384 7 cars, 2 trucks, 28.7ms
video 1/1 (frame 18/1626) /content/Driving in Dubai – How to Exit Duba

## Step 6: Save to Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')
!cp driving_perception_output.mp4 /content/drive/MyDrive/driving_perception_output.mp4

Mounted at /content/drive
